<a href="https://colab.research.google.com/github/Netrahoni/FlyRankAi-Intern-work-Files/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Netrahoni/FlyRankAi-Intern-work-Files/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Paper Finding 1: "Refreshing content older than 365 days yields a 45% greater traffic increase than refreshing recent content."

My Methodology Question: Where exactly does the 'traffic increase' label come from? Was it measured over a fixed, identical window (e.g., 30 days post-refresh) for all URLs, and does the validation design control for macro seasonality or core algorithmic updates that may have inflated that specific time period?

Paper Finding 2: "Content with AI-generated sections ranks 15% lower on average."

My Methodology Question: Does the validation design actually support this as a causal claim? Because this is observational data, how do we know the AI caused the drop, rather than lower-ranking, lower-budget sites simply being more likely to rely heavily on AI shortcuts?

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

When we split data randomly, URLs from the same domain leak into both the training and test sets. By switching to a grouped split based on client_id, we force the model to predict on entirely unseen domains, offering a more honest, real-world evaluation.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

df = pd.read_csv('content_refresh_anonymized.csv')

features = ['search_volume', 'word_count', 'impressions_90d', 'content_age_days', 'days_since_last_update']
target = 'trend_pct'
df_clean = df.dropna(subset=features + [target, 'client_id'])

X = df_clean[features]
y = df_clean[target]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y, test_size=0.2, random_state=42)
rf_random = RandomForestRegressor(n_estimators=50, random_state=42).fit(X_train_r, y_train_r)
mae_random = mean_absolute_error(y_test_r, rf_random.predict(X_test_r))

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df_clean['client_id']))
X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]

rf_grouped = RandomForestRegressor(n_estimators=50, random_state=42).fit(X_train_g, y_train_g)
mae_grouped = mean_absolute_error(y_test_g, rf_grouped.predict(X_test_g))

print(f"Random Split MAE (Overconfident): {mae_random:.2f}")
print(f"Grouped Split MAE (Honest): {mae_grouped:.2f}")

Random Split MAE (Overconfident): 71.49
Grouped Split MAE (Honest): 113.73


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

If we include impressions_last_30d and impressions_prev_30d as features to predict trend_pct, the model isn't learning content signals—it is just reverse-engineering the target's math equation. These must be dropped to prevent target leakage.

In [4]:
leakage_check = df_clean.select_dtypes(include='number').corr()[target].sort_values(ascending=False)
print("Top correlated features (Warning: values near 1.0 or -1.0 indicate severe leakage):")
print(leakage_check.head(5))

Top correlated features (Warning: values near 1.0 or -1.0 indicate severe leakage):
trend_pct               1.000000
impressions_last_30d    0.070248
ctr                     0.030143
clicks_last_30d         0.022407
sessions_last_30d       0.021431
Name: trend_pct, dtype: float64


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original (Overconfident): "The model accurately predicts the exact traffic trend percentage for any refreshed article based on its word count and age."

Rewritten (Safe): "The model provides directional decision-support, identifying observed content attributes (like word count and age) that have historically correlated with positive traffic trends following a refresh."

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.